<a href="https://colab.research.google.com/github/Ishalllll/preconception-stunting-risk-screening/blob/main/notebooks/data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pygrowup

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pygrowup: filename=pygrowup-0.8.2-py3-none-any.whl size=450766 sha256=221d8a877243d0eb06a8075841aa5cd96da88a67b9017549f3d78c1511051344
  Stored in directory: /root/.cache/pip/wheels/d3/1b/74/8f60c3253371ad9d204a34afa9be5e318041bcb238e75a7b84
Successfully built pygrowup


In [2]:
import os
import pandas as pd
import numpy as np

DATA_DIR = "."
FILE_ROS = "bk_ar1.csv"    # roster: umur, jenis kelamin, tanggal lahir, baris ibu
FILE_US  = "bus_us.csv"    # pengukuran fisik: US04 tinggi, US05 metode, US06 berat
FILE_KR  = "b2_kr.csv"     # karakteristik rumah, WASH
FILE_KRK = "bk_krk.csv"    # limbah, sampah

UMUR_MIN_BLN, UMUR_MAX_BLN = 0, 59


def baca_csv(nama, wajib=True):
    path = os.path.join(DATA_DIR, nama)
    if not os.path.exists(path):
        if wajib:
            raise FileNotFoundError(f"'{path}' tidak ada. Isi folder: {os.listdir(DATA_DIR)}")
        print(f"  [-] {nama} tidak ada, dilewati (opsional)")
        return None
    df = pd.read_csv(path, dtype=str)          # semua teks dulu -> ID aman
    df.columns = [c.lower().strip() for c in df.columns]
    return df

def angka(seri):
    return pd.to_numeric(seri, errors="coerce")

In [3]:
print("=" * 72)
print("Baca file")
print("=" * 72)

ros = baca_csv(FILE_ROS)
us  = baca_csv(FILE_US)
kr  = baca_csv(FILE_KR,  wajib=False)
krk = baca_csv(FILE_KRK, wajib=False)

print(f"  Roster    : {ros.shape[0]:,} baris x {ros.shape[1]} kolom")
print(f"  US (fisik): {us.shape[0]:,} baris x {us.shape[1]} kolom")

Baca file
  Roster    : 89,382 baris x 57 kolom
  US (fisik): 48,139 baris x 143 kolom


In [4]:
print("\n" + "=" * 72)
print("kohort balita, umur presisi bulanan")
print("=" * 72)
# HAZ butuh umur dalam BULAN, bukan tahun. Umur tahunan (ar09) terlalu kasar:
# anak 24 bulan dan 35 bulan sama-sama "2 tahun", padahal kurva WHO beda jauh.
# Maka umur dihitung dari tanggal lahir + tanggal wawancara.

ros["pidlink"] = ros["pidlink"].astype(str).str.strip()
ros["hhid14"]  = ros["hhid14"].astype(str).str.strip()
ros["_pid14i"] = angka(ros["pid14"])
ros["_umur_th"] = angka(ros["ar09"])          # umur tahunan, untuk pembanding kasar
ros["_sex"]     = angka(ros["ar07"])          # 1 = laki-laki, 3 = perempuan (cek!)
ros["_ibu"]     = angka(ros["ar11"])          # nomor baris ibu
ros["_ayah"]    = angka(ros["ar10"])

# tanggal lahir
lahir_thn = angka(ros.get("ar08yr"))
lahir_bln = angka(ros.get("ar08mth"))
# bersihkan kode missing (tahun mustahil, bulan di luar 1-12)
lahir_thn = lahir_thn.where((lahir_thn > 1990) & (lahir_thn < 2020))
lahir_bln = lahir_bln.where((lahir_bln >= 1) & (lahir_bln <= 12))

# tanggal wawancara: cari kolomnya, kalau tidak ada pakai asumsi
kol_ivw_bln = next((c for c in ros.columns if "ivwmth" in c), None)
kol_ivw_thn = next((c for c in ros.columns if "ivwyr" in c or "ivwyear" in c), None)
print(f"  Kolom tanggal wawancara ditemukan : bulan={kol_ivw_bln}, tahun={kol_ivw_thn}")

if kol_ivw_bln and kol_ivw_thn:
    ivw_bln = angka(ros[kol_ivw_bln]).where(lambda s: (s >= 1) & (s <= 12))
    ivw_thn = angka(ros[kol_ivw_thn]).where(lambda s: (s > 2013) & (s < 2017))
else:
    # fallback: IFLS-5 dilapangkan akhir 2014 sampai awal 2015. Pakai titik tengah.
    print("  [!] Tanggal wawancara tidak ketemu. Pakai asumsi Desember 2014.")
    print("      Ini menambah galat sampai beberapa bulan. Cari kolomnya kalau bisa.")
    ivw_bln = pd.Series(12, index=ros.index)
    ivw_thn = pd.Series(2014, index=ros.index)

ros["_umur_bln"] = (ivw_thn - lahir_thn) * 12 + (ivw_bln - lahir_bln)
ros["_umur_bln"] = ros["_umur_bln"].where(ros["_umur_bln"] >= 0)

balita = ros[(ros["_umur_bln"] >= UMUR_MIN_BLN) & (ros["_umur_bln"] <= UMUR_MAX_BLN)].copy()
print(f"  Balita 0-59 bulan (dari tanggal lahir) : {len(balita):,}")

# pembanding kasar dari umur tahunan, untuk deteksi anomali
kasar = ros[(ros["_umur_th"] >= 0) & (ros["_umur_th"] <= 4)]
print(f"  Pembanding, umur tahunan 0-4          : {len(kasar):,}")
if len(balita) > 0 and abs(len(balita) - len(kasar)) / len(kasar) > 0.25:
    print("  [!] Selisih dua cara hitung > 25%. Periksa tanggal lahir & wawancara.")

print(f"  Sebaran jenis kelamin (ar07) : {balita['_sex'].value_counts().to_dict()}")


kohort balita, umur presisi bulanan
  Kolom tanggal wawancara ditemukan : bulan=None, tahun=None
  [!] Tanggal wawancara tidak ketemu. Pakai asumsi Desember 2014.
      Ini menambah galat sampai beberapa bulan. Cari kolomnya kalau bisa.
  Balita 0-59 bulan (dari tanggal lahir) : 6,042
  Pembanding, umur tahunan 0-4          : 6,197
  Sebaran jenis kelamin (ar07) : {1.0: 3113, 3.0: 2929}


In [5]:
print("\n" + "=" * 72); print("TAHAP 2 : Sambungkan antropometri"); print("=" * 72)

us["pidlink"] = us["pidlink"].astype(str).str.strip()
us["_tinggi"] = angka(us.get("us04"))     # cm
us["_metode"] = angka(us.get("us05"))     # 1 = berdiri, 3 = berbaring
us["_berat"]  = angka(us.get("us06"))     # kg

# buang nilai mustahil (kode missing biasanya 999 dsb)
us["_tinggi"] = us["_tinggi"].where((us["_tinggi"] > 30) & (us["_tinggi"] < 150))
us["_berat"]  = us["_berat"].where((us["_berat"] > 1) & (us["_berat"] < 60))

kohort = balita.merge(
    us[["pidlink", "_tinggi", "_metode", "_berat"]], on="pidlink", how="left"
)
n_tinggi = kohort["_tinggi"].notna().sum()
print(f"  Balita dengan tinggi/panjang valid : {n_tinggi:,} dari {len(kohort):,}  ({n_tinggi/len(kohort)*100:.1f}%)")
print(f"  Balita dengan berat valid          : {kohort['_berat'].notna().sum():,}")
print(f"  Metode ukur (us05)                 : {kohort['_metode'].value_counts().to_dict()}")
print("     1 = berdiri (height), 3 = berbaring (recumbent length)")
print("     Standar WHO: <24 bulan pakai length, >=24 bulan pakai height.")
print("     Kalau metodenya tidak sesuai umur, ada koreksi 0,7 cm. Tangani saat")
print("     membangun dataset asli, bukan di skrip kelayakan ini.")

siap = kohort[kohort["_tinggi"].notna()].copy()
print(f"  >> Balita siap dihitung HAZ : {len(siap):,}   [GERBANG #1]")


TAHAP 2 : Sambungkan antropometri
  Balita dengan tinggi/panjang valid : 5,258 dari 6,042  (87.0%)
  Balita dengan berat valid          : 5,352
  Metode ukur (us05)                 : {1.0: 3445, 3.0: 1816}
     1 = berdiri (height), 3 = berbaring (recumbent length)
     Standar WHO: <24 bulan pakai length, >=24 bulan pakai height.
     Kalau metodenya tidak sesuai umur, ada koreksi 0,7 cm. Tangani saat
     membangun dataset asli, bukan di skrip kelayakan ini.
  >> Balita siap dihitung HAZ : 5,258   [GERBANG #1]


In [6]:
print("\n" + "=" * 72); print("TAHAP 3 : Hitung HAZ dan prevalensi stunting"); print("=" * 72)
# HAZ = z-score tinggi menurut umur, mengacu WHO Child Growth Standards.
# Rumusnya butuh tabel LMS resmi WHO, jadi dipakai library, bukan diperkirakan.

prevalensi = float("nan")
n_stunting = 0
try:
    from pygrowup import Calculator
    calc = Calculator(adjust_height_data=False, adjust_weight_scores=False,
                      include_cdc=False, logger_name="pg", log_level="CRITICAL")

    def hitung_haz(baris):
        # pygrowup: sex 'M'/'F', umur dalam bulan, tinggi cm
        try:
            sx = "M" if baris["_sex"] == 1 else "F"
            return float(calc.lhfa(baris["_tinggi"], baris["_umur_bln"], sx))
        except Exception:
            return np.nan

    siap["_haz"] = siap.apply(hitung_haz, axis=1)
    valid = siap["_haz"].notna()
    print(f"  HAZ berhasil dihitung untuk : {valid.sum():,} anak")

    # WHO: HAZ di luar -6..+6 dianggap implausible dan dibuang
    masuk_akal = siap["_haz"].between(-6, 6)
    print(f"  HAZ di luar -6..+6 (dibuang): {(valid & ~masuk_akal).sum():,}")
    bersih = siap[valid & masuk_akal]

    n_stunting = int((bersih["_haz"] < -2).sum())
    prevalensi = n_stunting / len(bersih) * 100 if len(bersih) else float("nan")
    print(f"  Sampel HAZ bersih  : {len(bersih):,}")
    print(f"  Stunting (HAZ<-2)  : {n_stunting:,}  ({prevalensi:.2f}%)   [GERBANG #2]")
    print(f"  Pembanding Novalina: 17,04%")
    print(f"  Sebaran HAZ        : min={bersih['_haz'].min():.2f}  "
          f"median={bersih['_haz'].median():.2f}  max={bersih['_haz'].max():.2f}")
except ImportError:
    print("  [-] pygrowup belum terpasang, HAZ dilewati.")
    print("      Jalankan: pip install pygrowup")
    print("      Gerbang lain di bawah tetap terjawab tanpa ini.")
except Exception as e:
    print(f"  [!] HAZ gagal dihitung: {e}")


TAHAP 3 : Hitung HAZ dan prevalensi stunting
  HAZ berhasil dihitung untuk : 5,258 anak
  HAZ di luar -6..+6 (dibuang): 92
  Sampel HAZ bersih  : 5,166
  Stunting (HAZ<-2)  : 1,484  (28.73%)   [GERBANG #2]
  Pembanding Novalina: 17,04%
  Sebaran HAZ        : min=-5.99  median=-1.18  max=5.97


In [7]:
print("\n" + "=" * 72); print("TAHAP 4 : Linkage anak ke ibu"); print("=" * 72)
# Cara kerjanya: AR11 anak = nomor baris ibu = pid14 ibu, di hhid14 yang sama.
# Kode 51 (ibu tak serumah) dan 52 (meninggal) otomatis gugur karena tidak ada
# orang dengan pid14 sebesar itu.

cari_ibu = ros[["hhid14", "_pid14i", "pidlink"]].rename(
    columns={"_pid14i": "_ibu", "pidlink": "pidlink_ibu"})
tsmb = siap.merge(cari_ibu, on=["hhid14", "_ibu"], how="left")
n_ibu = tsmb["pidlink_ibu"].notna().sum()
print(f"  Balita yang ibunya ada di roster : {n_ibu:,} dari {len(siap):,}  ({n_ibu/len(siap)*100:.1f}%)")

cari_ayah = ros[["hhid14", "_pid14i", "pidlink"]].rename(
    columns={"_pid14i": "_ayah", "pidlink": "pidlink_ayah"})
tsmb2 = siap.merge(cari_ayah, on=["hhid14", "_ayah"], how="left")
print(f"  Balita yang ayahnya ada di roster: {tsmb2['pidlink_ayah'].notna().sum():,}")
print("  Catatan: ini baru 'ibu ada di rumah tangga'. Apakah ibunya juga responden")
print("  Buku 3A (untuk pendidikan & usia menikah) perlu dicek terpisah dengan b3a.")


TAHAP 4 : Linkage anak ke ibu
  Balita yang ibunya ada di roster : 5,174 dari 5,258  (98.4%)
  Balita yang ayahnya ada di roster: 4,731
  Catatan: ini baru 'ibu ada di rumah tangga'. Apakah ibunya juga responden
  Buku 3A (untuk pendidikan & usia menikah) perlu dicek terpisah dengan b3a.


In [8]:
print("\n" + "=" * 72); print("TAHAP 5 : Penemuan variabel WASH"); print("=" * 72)
# Nama kolom WASH belum bisa dipastikan dari codebook yang ada, jadi skrip ini
# MENCETAK daftar kolom, bukan menebak. Lihat hasilnya, lalu tentukan sendiri.

def tampilkan_kolom(df, nama):
    if df is None:
        return
    print(f"\n  --- {nama} : {df.shape[0]:,} baris x {df.shape[1]} kolom ---")
    print(f"  Semua kolom: {list(df.columns)}")

tampilkan_kolom(kr,  FILE_KR)
tampilkan_kolom(krk, FILE_KRK)
print("\n  Cari kolom yang berhubungan dengan: sumber air minum, jamban/kakus,")
print("  pembuangan limbah, jenis lantai, listrik. Cocokkan dengan codebook Buku 2.")


TAHAP 5 : Penemuan variabel WASH

  --- b2_kr.csv : 15,185 baris x 75 kolom ---
  Semua kolom: ['hhid14_9', 'kr03', 'kr04ax', 'kr05ax', 'kr11', 'kr13', 'kr13a', 'kr13b', 'kr14', 'kr16', 'kr17', 'kr17b', 'kr18', 'kr20', 'kr21', 'kr22', 'kr23', 'kr24', 'kr24a', 'kr24b1', 'kr24b2', 'kr24b3', 'kr24b4', 'kr24b5', 'kr24b6', 'kr24b7', 'kr24b8', 'kr24b9', 'kr24cx', 'kr25', 'kr26', 'kr26ax', 'kr26a', 'kr27', 'kr27a', 'kr27b', 'kr27d', 'kr27e', 'kr27f', 'kr27g', 'kr27h', 'kr27i', 'kr27j1x', 'kr27j1a', 'kr27j1b', 'kr27j1c', 'kr27j1d', 'kr27j1e', 'kr27j2x', 'kr27j2a', 'kr27j2b', 'kr27j2c', 'kr27j2d', 'kr27j2e', 'kr27j3x', 'kr27j3a', 'kr27j3b', 'kr27j3c', 'kr27j3d', 'kr27j3e', 'kr27j4x', 'kr27j4a', 'kr27j4b', 'kr27j4c', 'kr27j4d', 'kr27j4e', 'kr27k', 'hhid14', 'kr04a', 'kr05a', 'kr24c', 'version', 'module', 'kr15', 'kr19']

  --- bk_krk.csv : 15,921 baris x 19 kolom ---
  Semua kolom: ['hhid14_9', 'krk01', 'krk02a', 'krk02b', 'krk02c', 'krk02d', 'krk02e', 'krk02f', 'krk02g', 'krk02h', 'krk02i', 'k

In [9]:
print("\n" + "=" * 72); print("TAHAP 6 : RINGKASAN GERBANG  <-- BACA INI"); print("=" * 72)
print(f"  #1  Balita 0-59 bln dengan tinggi valid : {len(siap):,}")
print(f"      (Novalina dapat 1.561 rekaman complete-case)")
if not np.isnan(prevalensi):
    print(f"  #2  Prevalensi stunting                 : {prevalensi:.2f}%  (Novalina 17,04%)")
else:
    print(f"  #2  Prevalensi stunting                 : belum dihitung, pasang pygrowup")
print(f"  #3  Balita yang ibunya ada di rumah     : {n_ibu:,}")
print(f"  #4  WASH                                : lihat daftar kolom di TAHAP 5")


TAHAP 6 : RINGKASAN GERBANG  <-- BACA INI
  #1  Balita 0-59 bln dengan tinggi valid : 5,258
      (Novalina dapat 1.561 rekaman complete-case)
  #2  Prevalensi stunting                 : 28.73%  (Novalina 17,04%)
  #3  Balita yang ibunya ada di rumah     : 5,174
  #4  WASH                                : lihat daftar kolom di TAHAP 5
